# Workshop 4. Comparing two groups

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture04/workshop04.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

Task 1 is the comparison of Copenhagen and Aarhus delivery times from the lecture in code, fully worked: run it, answer
the questions and change one thing. Tasks 2 to 4 apply the same steps to new questions, starting from an empty cell:
late deliveries by warehouse (two proportions), the airline survey (paired ratings), and the spread of the two
carriers (two variances). Then do the short problems, which use small numbers and no data, much like a question at the
oral exam. The extension tasks are for those who have time left.

Under each core task there are three hints: the first names the statistical step, the second names the functions,
and the third gives the structure of the code as comments. Try the task before you open them. The code patterns
below, the slides and the lecture notebook are your main references. You are welcome to use an LLM, but every task
can be solved without one, and you should be able to explain the statistics in your answers yourself.

In every task, decide first whether the samples are paired or independent. Then write H<sub>0</sub> and
H<sub>1</sub> before you compute anything, and decide from the question whether the test is one-sided or two-sided.

## Toolbox

The functions you need today. The lecture notebook shows all of them in use.

| Function | What it does |
|---|---|
| `df["col"].mean()`, `.std(ddof=1)`, `.var(ddof=1)` | Mean, standard deviation and variance of a column, dividing by n − 1 |
| `len(x)`, `x.sum()` | Number of observations, number of 1s in a 0/1 column |
| `df.loc[df["col"] == "value", "other"]` | The column `other` for the rows where `col` has a given value |
| `df["a"] - df["b"]` | The difference in each row: one column of changes for paired data |
| `np.sqrt(x)` | Square root |
| `stats.t.ppf(1 - alpha, df=...)`, `stats.f.ppf(1 - alpha, nu1, nu2)` | One-sided critical values: α above them |
| `stats.t.ppf(1 - alpha / 2, df=...)`, `stats.norm.ppf(1 - alpha / 2)`, `stats.f.ppf(1 - alpha / 2, nu1, nu2)` | Two-sided critical values: α/2 above them |
| `stats.t.sf(t, df=...)`, `stats.norm.sf(z)`, `stats.f.sf(F, nu1, nu2)` | Area above a value: the p-value when H<sub>1</sub> has >. `cdf` gives the area below |
| `stats.ttest_rel(x, y, alternative=...)` | Paired t test of x − y, as a check |
| `stats.ttest_ind(x1, x2, equal_var=False, alternative=...)` | Welch's t test for two independent means, as a check. Without `equal_var=False` it is the pooled test |
| `stats.skew(x)`, `plt.hist(x, bins=15)` | Skewness and histogram, to check normality |

## Code patterns

Four complete patterns for today, one for each test. Copy a pattern into a task cell, replace `df`, the column names
and the group names, and choose the p-value line that matches your H<sub>1</sub>. The patterns are templates; they do
not run as they stand.

**Paired t test (the same units measured twice)**

```python
d = df["after"] - df["before"]               # one difference per row
n = len(d)
t_stat = d.mean() / (d.std(ddof=1) / np.sqrt(n))
p = stats.t.sf(t_stat, df=n - 1)             # H1: mu_d > 0
# H1: mu_d < 0   p = stats.t.cdf(t_stat, df=n - 1)
# H1: mu_d != 0  p = 2 * stats.t.sf(abs(t_stat), df=n - 1), critical values -/+ stats.t.ppf(1 - alpha / 2, df=n - 1)
print(f"d_bar = {d.mean():.3f}, t = {t_stat:.3f}, p = {p:.4f}")
```

**t test for two independent means (Welch)**

```python
x1 = df.loc[df["group"] == "first", "column"]
x2 = df.loc[df["group"] == "second", "column"]
n1, n2 = len(x1), len(x2)
v1 = x1.var(ddof=1) / n1
v2 = x2.var(ddof=1) / n2
se = np.sqrt(v1 + v2)                        # variances add
t_stat = (x1.mean() - x2.mean()) / se
df_w = (v1 + v2)**2 / (v1**2 / (n1 - 1) + v2**2 / (n2 - 1))
p = stats.t.sf(t_stat, df=df_w)              # H1: mu1 - mu2 > 0; two-sided: 2 * stats.t.sf(abs(t_stat), df=df_w)
print(f"t = {t_stat:.3f}, df = {df_w:.2f}, p = {p:.4f}")
```

**z test for two proportions (pooled share, as in Newbold)**

```python
x1 = df.loc[df["group"] == "first", "column"]     # a 0/1 column
x2 = df.loc[df["group"] == "second", "column"]
n1, n2 = len(x1), len(x2)
p0 = (x1.sum() + x2.sum()) / (n1 + n2)            # pooled share
print(n1 * p0 * (1 - p0), n2 * p0 * (1 - p0))     # the condition: both above 5
z = (x1.mean() - x2.mean()) / np.sqrt(p0 * (1 - p0) * (1 / n1 + 1 / n2))
p = stats.norm.sf(z)                              # H1: P1 > P2; two-sided: 2 * stats.norm.sf(abs(z))
print(f"z = {z:.3f}, p = {p:.4f}")
```

**F test for two variances (needs normal populations)**

```python
F = x1.var(ddof=1) / x2.var(ddof=1)          # put the larger variance on top
nu1, nu2 = len(x1) - 1, len(x2) - 1          # numerator first
crit = stats.f.ppf(1 - alpha, nu1, nu2)      # one-sided; two-sided: stats.f.ppf(1 - alpha / 2, nu1, nu2)
p = stats.f.sf(F, nu1, nu2)                  # one-sided; two-sided: 2 * stats.f.sf(F, nu1, nu2)
print(f"F = {F:.3f}, crit = {crit:.3f}, p = {p:.4f}")
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")          # the study of 200 orders
airline = pd.read_csv(url + "airline_survey.csv")        # the airline passenger survey from workshop 3
alpha = 0.05

## Core tasks

### Task 1. Copenhagen against Aarhus in code (worked)

Do orders shipped from Copenhagen take longer on average than orders shipped from Aarhus? The orders in the two groups
are different orders, so the samples are independent. The hypotheses are H<sub>0</sub>: μ<sub>1</sub> − μ<sub>2</sub>
≤ 0 and H<sub>1</sub>: μ<sub>1</sub> − μ<sub>2</sub> > 0, with group 1 Copenhagen and group 2 Aarhus. The cell below
runs Welch's test exactly as in the lecture. Run it and answer the questions.

1. Which lines compute the standard error? Why are the two parts added, and why is the result larger than either
group's own standard error?
2. Is Carrier B slower than Carrier A? Change the first two lines to `column = "carrier"` and
`group1, group2 = "Carrier B", "Carrier A"` and run the cell again. What do you conclude?
3. Write the result for the two warehouses in one or two sentences for a manager at RetailCorp. Include what the
comparison cannot show.

In [ ]:
column = "warehouse"                          # question 2: "carrier"
group1, group2 = "Copenhagen", "Aarhus"       # question 2: "Carrier B", "Carrier A"

x1 = sample.loc[sample[column] == group1, "delivery_hours"]
x2 = sample.loc[sample[column] == group2, "delivery_hours"]
n1, n2 = len(x1), len(x2)
print(f"{group1}: n = {n1}, mean = {x1.mean():.3f}, s = {x1.std(ddof=1):.3f}")
print(f"{group2}: n = {n2}, mean = {x2.mean():.3f}, s = {x2.std(ddof=1):.3f}")

v1 = x1.var(ddof=1) / n1                      # s1^2 / n1
v2 = x2.var(ddof=1) / n2                      # s2^2 / n2
se = np.sqrt(v1 + v2)                         # variances add
t_stat = (x1.mean() - x2.mean()) / se
df_w = (v1 + v2)**2 / (v1**2 / (n1 - 1) + v2**2 / (n2 - 1))   # Welch degrees of freedom
t_crit = stats.t.ppf(1 - alpha, df=df_w)
p = stats.t.sf(t_stat, df=df_w)               # area above t, since H1 has >
print(f"se = {se:.4f}, t = {t_stat:.3f}, df = {df_w:.2f}, crit = {t_crit:.3f}, p = {p:.3f}")

**Your answer:**

### Task 2. Are more Copenhagen orders late?

An order is late if it takes more than 48 hours (column `late`, 1 for late). Operations suspects that a larger share
of Copenhagen orders is late than of Aarhus orders. Test this at α = 0.05 with the 200 orders.

**Answer in words.** Are the samples paired or independent? State H<sub>0</sub> and H<sub>1</sub>, check the
condition, and give the decision and the result for operations.

<details>
<summary>Hint 1</summary>

This is the z test for two proportions with the pooled share p̂<sub>0</sub>, one-sided: H<sub>0</sub>:
P<sub>Cph</sub> − P<sub>Aar</sub> ≤ 0 and H<sub>1</sub>: P<sub>Cph</sub> − P<sub>Aar</sub> > 0. The orders in the two
groups are different orders, so the samples are independent.
</details>

<details>
<summary>Hint 2</summary>

`sample.loc[sample["warehouse"] == "Copenhagen", "late"]`, `.sum()`, `.mean()`, `len`, `np.sqrt`,
`stats.norm.ppf(1 - alpha)` and `stats.norm.sf`. See the third code pattern.
</details>

<details>
<summary>Hint 3</summary>

```python
# select the late column for Copenhagen and for Aarhus
# n1, n2 and the two sample shares
# the pooled share: all late orders divided by all orders
# the condition n * p0 * (1 - p0) for each group
# the standard error with p0, then z
# the critical value and the p-value (area above z)
```
</details>

**Your answer:**

### Task 3. The airline survey: check-in against baggage handling

In the airline survey each passenger rated several services from 1 to 5. Do passengers rate check-in service and
baggage handling differently on average? Use the columns `Checkin service` and `Baggage handling`, which have no zeros
("not rated"). Test at α = 0.05.

**Answer in words.** Are the samples paired or independent, and why? State H<sub>0</sub> and H<sub>1</sub>, the
decision, and the result for the airline's service manager. How large is the average difference on a scale from 1 to
5?

<details>
<summary>Hint 1</summary>

Each passenger gave both ratings, so the samples are paired. Compute the difference for each passenger and use the
paired t test, two-sided: H<sub>0</sub>: μ<sub>d</sub> = 0 and H<sub>1</sub>: μ<sub>d</sub> ≠ 0.
</details>

<details>
<summary>Hint 2</summary>

`airline["Checkin service"] - airline["Baggage handling"]`, `.mean()`, `.std(ddof=1)`, `len`, `np.sqrt`,
`stats.t.ppf(1 - alpha / 2, df=n - 1)` and `2 * stats.t.sf(abs(t), df=n - 1)`. Check with
`stats.ttest_rel(airline["Checkin service"], airline["Baggage handling"])`.
</details>

<details>
<summary>Hint 3</summary>

```python
# d = check-in rating minus baggage rating, one value per passenger
# n, the mean and the standard deviation of d
# t = mean / (sd / sqrt(n))
# the two-sided critical value and the two-sided p-value
# print n, the mean of d, t, the critical value and p
```
</details>

**Your answer:**

### Task 4. Do the two carriers differ in spread?

A larger spread in delivery times makes deliveries less predictable. Do the delivery times of Carrier A and Carrier B
differ in spread, in either direction? Test at α = 0.05. The test assumes normal populations, so look at the two
histograms first.

**Answer in words.** State H<sub>0</sub> and H<sub>1</sub>. Is the normality assumption reasonable? Which variance
did you put on top, and why? Give the decision and what it tells RetailCorp, and what it does not tell.

<details>
<summary>Hint 1</summary>

The F test for two variances, two-sided: H<sub>0</sub>: σ<sub>A</sub>² = σ<sub>B</sub>² and H<sub>1</sub>:
σ<sub>A</sub>² ≠ σ<sub>B</sub>². As in Newbold, put the larger sample variance on top, compare F with the critical
value at α/2, and double the area above F for the p-value.
</details>

<details>
<summary>Hint 2</summary>

`sample.loc[sample["carrier"] == "Carrier A", "delivery_hours"]`, `plt.hist`, `stats.skew`, `.var(ddof=1)`,
`stats.f.ppf(1 - alpha / 2, nu1, nu2)` and `2 * stats.f.sf(F, nu1, nu2)`. See the fourth code pattern.
</details>

<details>
<summary>Hint 3</summary>

```python
# select the delivery times for each carrier
# histograms and skewness for both
# the two sample variances: which is larger?
# F = larger variance / smaller variance, with nu1 and nu2 in the same order
# the critical value at alpha / 2 and the two-sided p-value
```
</details>

**Your answer:**

## Short problems

These problems use small numbers and no data. Solve them with the formulas from the slides, and use Python only to
look up critical values and p-values, or a table if you prefer. Write your reasoning in words, as you would explain it
at the exam.

### Problem 1. A new shelf display

A chain tries a new shelf display in 8 of its stores. Weekly sales (in thousand DKK) before and after the change:

| Store | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| Before | 52 | 48 | 61 | 45 | 57 | 50 | 63 | 54 |
| After | 55 | 49 | 64 | 44 | 60 | 53 | 66 | 55 |

1. Are the samples paired or independent? State H<sub>0</sub> and H<sub>1</sub> for "sales went up".
2. Compute the differences, d̄, s<sub>d</sub> and t. Use t<sub>7, 0.05</sub> = 1.895 to make the decision.
3. What does the test assume about the differences, and why does it matter more here than in the lecture?

<details>
<summary>Hint 1</summary>

The same stores are measured twice: paired. d = after − before for each store, then t = d̄ / (s<sub>d</sub>/√8) with
7 degrees of freedom.
</details>

<details>
<summary>Hint 2</summary>

`np.array([...])`, `.mean()`, `.std(ddof=1)`, `stats.t.sf(t, df=7)`.
</details>

**Your answer:**

### Problem 2. Two training methods

A company trains new sales staff with two methods and compares their test scores. Method 1: 12 employees, mean 74,
standard deviation 8. Method 2: 10 employees, mean 67, standard deviation 9. The two groups are different employees,
and the scores are roughly normal. The company assumes that the population variances are equal.

1. State H<sub>0</sub> and H<sub>1</sub> for "the methods differ".
2. Compute the pooled variance and t. Use t<sub>20, 0.025</sub> = 2.086 to make the decision.
3. Why are the degrees of freedom 20?

<details>
<summary>Hint 1</summary>

Independent samples, pooled t test (variances assumed equal), two-sided. s<sub>p</sub>² = ((n<sub>1</sub> −
1)s<sub>1</sub>² + (n<sub>2</sub> − 1)s<sub>2</sub>²) / (n<sub>1</sub> + n<sub>2</sub> − 2).
</details>

<details>
<summary>Hint 2</summary>

`np.sqrt`, `stats.t.ppf(0.975, df=20)`, `2 * stats.t.sf(abs(t), df=20)`. As a check:
`stats.ttest_ind_from_stats(74, 8, 12, 67, 9, 10)`.
</details>

**Your answer:**

### Problem 3. Two webshop layouts

A webshop shows layout 1 to 300 random visitors and layout 2 to 300 others. 45 visitors buy with layout 1 and 69 with
layout 2.

1. State H<sub>0</sub> and H<sub>1</sub> for "the layouts differ", and check the condition.
2. Compute the pooled share, z and the p-value. What is the decision at α = 0.05?
3. What can the webshop conclude about the cause of the difference, and why?

<details>
<summary>Hint 1</summary>

Two proportions, independent, two-sided. p̂<sub>0</sub> = (45 + 69)/600.
</details>

<details>
<summary>Hint 2</summary>

`np.sqrt(p0 * (1 - p0) * (1 / 300 + 1 / 300))`, `2 * stats.norm.sf(abs(z))`.
</details>

**Your answer:**

### Problem 4. Two filling machines

Machine A fills 16 bags with a standard deviation of 3.0 g; machine B fills 21 bags with a standard deviation of
1.8 g. The weights are normal. Is machine A less precise than machine B?

1. State H<sub>0</sub> and H<sub>1</sub>.
2. Compute F and make the decision with F<sub>15, 20, 0.05</sub> = 2.203.
3. Why does the problem need to say that the weights are normal?

<details>
<summary>Hint 1</summary>

One-sided F test: H<sub>0</sub>: σ<sub>A</sub>² ≤ σ<sub>B</sub>², H<sub>1</sub>: σ<sub>A</sub>² > σ<sub>B</sub>². F
uses the variances, so square the standard deviations. Machine A on top, with 15 and 20 degrees of freedom.
</details>

<details>
<summary>Hint 2</summary>

`stats.f.ppf(0.95, 15, 20)` and `stats.f.sf(F, 15, 20)`.
</details>

**Your answer:**

### Problem 5. Paired or independent?

For each study, say whether the samples are paired or independent, and which test you would use.

1. The same 40 employees take a test before and after a course.
2. 200 customers in Jutland and 250 customers on Zealand rate a service from 1 to 10.
3. The share of defective items in a sample of 500 from machine A and 400 from machine B.
4. Twins: one twin in each pair gets a coaching program, the other does not, and their incomes are compared.
5. Whether filling weights vary more for one machine than for another.

**Your answer:**

## Extension tasks

### Extension 1. How much does pairing help?

In Task 3 you used the paired test. Analyze the same two airline columns as if they were two independent groups, with
the standard error √(s<sub>1</sub>²/n + s<sub>2</sub>²/n), and compare the standard error and t with Task 3. Then
compute the correlation between the two ratings with `airline["Checkin service"].corr(airline["Baggage handling"])`.

**Answer in words.** Why does pairing change the standard error much less here than in the RetailCorp survey in the
lecture, where the correlation between before and after was 0.94?

<details>
<summary>Hint 1</summary>

The independent standard error uses the variance of each column; the paired one uses the variance of the
differences. The variance of a difference is smaller when the two measurements move together.
</details>

<details>
<summary>Hint 2</summary>

`x1.var(ddof=1)`, `x2.var(ddof=1)`, `np.sqrt`, `.corr()`.
</details>

**Your answer:**

### Extension 2. Large samples in the airline file

1. Test whether the share of satisfied passengers (column `satisfaction` equal to "satisfied") differs between women
and men (column `Gender`), two-sided.
2. Test whether average seat comfort differs between women and men, two-sided, with Welch's test.

**Answer in words.** Compare the two results. What does a sample of 26,000 passengers do, and what does it not do?

<details>
<summary>Hint 1</summary>

Part 1 is a two-proportion z test with the pooled share; part 2 is Welch's t test. Both are two-sided.
</details>

<details>
<summary>Hint 2</summary>

`(airline["satisfaction"] == "satisfied")` gives a 0/1 column. `stats.ttest_ind(women, men, equal_var=False)`.
</details>

**Your answer:**

### Extension 3. When pooled and Welch disagree

A study compares a small group (n = 10, mean 52, standard deviation 12) with a large group (n = 60, mean 45, standard
deviation 4). Run the two-sided test with `stats.ttest_ind_from_stats(52, 12, 10, 45, 4, 60)`, first as written (the
default), then with `equal_var=False` added.

**Answer in words.** Compare the two results. Which version is the default, why do they differ so much here, and
which one would you report?

<details>
<summary>Hint 1</summary>

The default is the pooled test. Look at which group's variance dominates the pooled variance, and which group has the
larger variance.
</details>

<details>
<summary>Hint 2</summary>

`stats.ttest_ind_from_stats(mean1, std1, nobs1, mean2, std2, nobs2, equal_var=False)`.
</details>

**Your answer:**